# Datamine MODENV Process Example

This notebook demonstrates how to configure and run the **`modenv`** process wrapper in `dmstudio`.

## Process Description

## MODENV

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

The **MODENV** process analyses a block model and flags those blocks that when taken individually or with adjacent blocks satisfy the grade or economic value criteria within a volume defined by the minimum mining unit (MMU) dimensions.

The tailored interface for this process is called the **Mineable Reserves Optimizer** (MRO)

For the underground situation the MMU is the minimum stope size and for the open pit case it will be a selective mining unit.

The process will locate envelopes of economic mining units and provide a first pass evaluation of that portion of a geological model that would be economically mineable. No detailed consideration is given to the mine development required for access, ore haulage or support.

The process provides a result analogous to the floating cone method of pit optimization in open pit mine planning. It was designated the 'floating stope' method when first implemented in the **STPMOD** process as it was initially used for underground optimization. However the method has since been extended to include the open pit case as well. The effect of different MMU dimensions and cutoffs can be evaluated with multiple runs.

The input to **MODENV** includes many files, fields and parameters, and so a set of tailored dialogs is avaiable to facilitate data entry and allow multiple options for analysing the results. . The description below is for the **MODENV** process run using the standard Files, Fields, Parameters dialog. The **MODENV** process can also be included in macros and scripts.

* **Note** (The [ENVSEQ](<envseq.md>) process takes the envelope output model from **MODENV** and evaluates design alternatives and the economics of sequential development of envelopes. This is another feature of MRO.):

**Note** : MRO has its own help file and associated tutorial. Click **?** on the MRO screen.

### Optimization Method

The shape of an MMU is defined by a set of 3D rectangular model cells (sub-MMUs) with all cells being the same size. The input geological model is reblocked onto the sub-MMU size and the MMU is moved over the reblocked model in sub-MMU increments so that all possible MMU locations are evaluated. Different algorithms (Maximum, Minimum, Ranked) are then available for defining whether an MMU is economic and will be included in a mineable envelope.

A sub-MMU may form part of multiple MMUs. The criteria for defining the optimal MMU position is selected using parameter @**OPTIMISE** which gives the following options:

1. Maximum ore tonnes (equivalent to minimum included waste or dilution). If several MMUs have the same tonnes, then the MMU with the maximum grade is preferred.
2. Maximum head grade for the MMU.
3. Maximum contained metal for the MMU.
4. Maximum accumulated value, where the supplied field is expressed in currency value, for example, $USD

For cases (1) - (3) a grade or equivalent grade for polymetallic deposits must be defined using field *GRADE.

For case (4), the value field * **VALUE** should define the total cell value, to reflect mining and processing costs and the product value for the cell. Parameters @**DEFVALUE** , @**CUTOFF** and @**HDGRADE** must then be expressed as a value per unit volume. @**DEFVALUE** defines the default value for any unmodeled cell, and typically reflects the cost of mining a unit volume of waste. Both @**CUTOFF** and @**HDGRADE** parameters are normally set to zero to indicate that all blocks with a positive value are worth mining and any MMU with a positive value is acceptable for mining. Any density field or parameter supplied will be ignored and a density value of 1.0 is used internally within the process for volumetric calculations. The parameters @**MAXWASTE** and @**DILUTE** will still operate with 'waste' defined as blocks below the @**CUTOFF** parameter value.

Both a grade (* **GRADE**) and a value (* **VALUE**) field can be used simultaneously. Both fields will be reported but the only one field will be optimized as defined by @**OPTIMISE**.

### MMU Shape and Orientation

The shape and orientation of the MMU can be specified by one of several methods, providing a progressive increase in flexibility. The four methods are:

1. Defined by a 3D rectangular shaped MMU which is orthogonal to the axes of the input model. The method allows the possible MMU positions to be specified by defining the MMU origin, maximum position, and MMU increment in each of the three coordinate axis directions. The parameters are:

MMUSIZE[X|Y|Z] |  the dimensions of the MMU
---|---
MMUINC[X|Y|Z] |  the number of MMU increments within each MMU dimension
MMUMIN[X|Y|Z] | the minimum coordinates (bottom left corner) for an MMU
MMUMAX[X|Y|Z] |  the maximum coordinates (top right corner) for an MMU
2. The **MMUMIN** and **MMUMAX** parameters are optional. If they are not selected then the MMUMIN values are set equal to the origin of the input geological model and the **MMUMAX** to the coordinates of the top right corner of the model.

3. The parameters limit an MMU to a rectangular shape moving on an orthogonal grid. The MMU moves in steps of **MMUSIZE** / **MMUINC** (the sub-MMU size) along each coordinate axis. The smaller the sub-MMU (ie the larger the **MMUINC**) the more positions will be sampled thus improving the approximation to a floating MMU to optimize the envelope location. However in general for each direction there is no point in making the sub-MMU size smaller than the size of the minimum subcell in the input model in that direction. The larger the **MMUINC** parameters, the longer the processing time.
4. A non-rectangular shape can be generated by defining the MMU size in X and Y as a base dimension, and specifying a slope factor 1:n in each of the four horizontal orthogonal directions. Positive n implies an outward expansion and negative n an inward contraction. If all factors are positive and parameter **MMUSLOPI** =0 then the MMU represents an inverted truncated pyramid with a rectangular horizontal cross-section. If @**MMUSLOPI** =1 then the MMU will be a truncated cone shape with an elliptical horizontal cross-section. The required parameters are:

MMUSIZE[X|Y] |  The base dimensions of the MMU in the XY plane
---|---
MMUINC[X|Y] |  The number of MMU increments within the base dimension
MMUSIZE[Z] |  The height of the MMU
MMUINC[Z] |  The number of MMU increments within the MMU height
MMUMIN[X|Y|Z] |  The minimum coordinates (bottom left corner) for an MMU
MMUMAX[X|Y|Z] |  The maximum coordinates (top right corner) for an MMU
MMUSLOPE[N|S|E|W] |  MMU slope factor (1:n) in the N|S|E|W direction. The slope factor is a ratio of sub-MMUs extended horizontally to vertically ie 0.5 would be half a sub-MMU horizontally for each sub-MMU vertically. Note that this definition is the inverse of the normal slope definition but allows a vertical side to be expressed exactly as zero whereas the standard definition of slope would require an infinite value.
MMUSLOPI |  Flag to specify whether the MMU slope factors are to be interpolated in plan.
0= do not interpolate ie MMU is always rectangular in plan.
1= interpolate. ie MMU is elliptical in plan.
5. The **MMUMIN** and **MMUMAX** parameters are compulsory. All six parameters (**MMUMIN**[X|Y|Z] and **MMUMAX**[X|Y|Z]) must be defined for this method to be applied. The parameters cannot be used if the input model is rotated.
6. A rotation can be applied to methods 1 or 2 by including a definition for a Rotated Model as the MMU shape template model file. In these cases the limits of the MMUs are defined by the limits of the Rotated Model, and so the parameters **MMUMIN**[X|Y|Z] and **MMUMAX**[X|Y|Z] are not required.

7. User defined MMU shapes can be specified by providing an MMU shape template model file, &**SHAPE** , with cells supplied to model the MMU shape(s). None of the parameters **MMUSIZE**[X|Y|Z], **MMUMIN**[X|Y|Z], or **MMUMAX**[X|Y|Z] are required. As for methods 2 and 3 the limits of the MMUs are defined by the limits of the &**SHAPE** model file. The sub-MMU dimensions are calculated by dividing the parent cell size for &SHAPE by the MMU increment **MMUINC**[X|Y|Z]. The &**SHAPE** file may include both cells and subcells to define the MMU shape. If the dimensions of the subcells are not multiples of the sub-MMU size then all MMUs intersected by subcells will be used to define to shape. The actual location of the cells in the &SHAPE model is not important, only the relative position of the cells. The MMU shape will be floated to all possible positions within the model file unless restricted by the @**INMODEL** parameter.

If required the MMU shape template file can be a Rotated Model file to allow the MMU to float non-orthogonally to the block model axes. If a * **SHAPZONE** field is supplied in the input block model and in the MMU shape template file then different MMU shapes can be used in different mining areas. If not all values of the **SHAPZONE** field in the model have a corresponding shape definition in the &**SHAPE** file then the default MMU shape will be used. In the MMU shape template file the default shape will have a **SHAPZONE** value of blank for alphanumeric fields and - for numeric fields. Up to 50 zone values can be modeled. Where a block can be extracted by MMU shapes from different zones, the shape is selected by taking the MMU shape matching the block zone or else the MMU shape that would be least diluted by other material from other zones or the MMU with the greatest value as defined by the @**SHAPEMTH** parameter.

* **Important** (In all four methods of MMU specification, the MMU increment must be supplied. If an MMU shape is specified in the MMU shape template file the correct cell or subcell size to match the MMU increment should be selected.):

### Economic Parameters for MMUs

In addition to the geometric parameters defined in the previous section the following parameters are used to define an economic MMU.

Head Grade is the minimum grade of an economic MMU. It is defined either by the single parameter @**HDGRADE** or the input model file &IN can include field * **HDGRADE** , allowing the Head Grade to vary over the model volume.

Cutoff Grade differentiates between ore and waste and is defined by parameter @**CUTOFF**. It is used in the definition of a Minimum envelope as described below.

The maximum volume of waste (material below Cutoff) that can be included in an economic MMU is defined by the @**MAXWASTE** parameter. This is expressed as a proportion between 0 (no waste allowed) and 1 (any volume of waste allowed).

The @**DILUTE** parameter defines whether the grade of an MMU should be calculated as the average of all associated sub-MMUs (@**DILUTE** =1) or just the sub-MMUs above Cutoff (@DILUTE=0). The default is 1 which implies that ore and waste cannot be selectively mined within an MMU, which is the usual case.

### Pre-Defined Waste

Pre-defined waste volumes can optionally be modeled prior to generating the envelopes. The minimum dimension of pre-defined waste can be defined, and areas satisfying these criteria can either be excluded from the envelope, or used as a last resort to mine economic ore. If parameter @**PDWONLY** is set to 1 then only the pre-defined waste envelope will be output. If @**PWDONLY** is set to 0 then both pre-defined waste and MMU envelopes will be modeled. In order to use the pre-defined waste option, all three dimensions of the pre-defined waste volume (@**PDWSIZE**[X|Y|Z]) must be defined. The dimensions of the pre-defined waste should be defined in multiples of the sub-MMU size. If this is not the case they will be rounded to the nearest sub-MMU size.

The maximum value for pre-defined waste is specified using the @PDWGRADE or @PDWVALUE parameter. The maximum volume of ore (material above Cutoff) that can be included in pre-defined waste is defined by the @MAXORE parameter. This is expressed as a proportion between 0 (no ore allowed) and 1 (any volume of ore allowed).

### Constraining Envelopes

Specific areas of the model can be excluded from the optimization to constrain the MMU envelopes. Examples would include previously mined areas, blocks above the topography surface, or areas that have been excluded for support, access or poor ground conditions. Several methods are available:

1. File &**EXCLUDE** can include a list of values for a field that already exists in the input model to flag those blocks as unavailable for mining. An acceptable MMU cannot contain all or part of one of these blocks.
2. Where a **MINED** field is supplied with the input model, a non-zero value of the **MINED** field will prohibit an MMU including that block.
3. The MMU can be constrained to lie within blocks in the input model by setting parameter @**INMODEL** =1. An MMU will then only be considered if the MMU volume lies completely within modeled blocks. If @**INMODEL** =0 then an MMU can include unmodeled material that is assigned a default grade (@**DEFGRADE**), default value (@**DEFVALUE**) and default density (@**DENSITY**).

If the density of a block is set to zero then the block is considered as an air block instead of undefined waste.

### Mineable Envelopes

The process defines 'mineable envelopes', which are created by the accumulation of all best contiguous MMUs. With the @ENVTYPE parameter it is possible to define different types of envelopes:

0\. Maximum

1\. Minimum

2\. Ranked

The input model is reblocked into sub-MMU sized cells and each sub-MMU is analysed in turn. All possible MMUs which include that sub-MMU are evaluated and all sub-MMUs that lie within economic MMUs (grade greater than **Head Grade**) are flagged as part of the Maximum envelope. If a Maximum envelope has been defined (@**ENVTYPE** =0) then processing moves on and the next sub-MMU is analysed and so on until all sub-MMUs have been considered. This definition of the Maximum envelope means that several low grade sub-MMUs surrounding a group of high grade sub-MMUs may be assigned to a Maximum envelope even though the average grade of the total envelope may be below Head Grade.

If a Minimum (@**ENVTYPE** =1) or Ranked (@**ENVTYPE** =2) envelope has been selected then further processing is applied. For a Minimum envelope each sub-MMU is analysed in turn and if the sub-MMU is above Cutoff and it forms part of one or more economic MMUs then the best MMU as defined by @**OPTIMISE** is selected and all sub-MMUs within that best MMU are assigned to the Minimum envelope. In this way a sub-MMU that was initially assigned as Maximum may be reassigned as Minimum.

For a Ranked envelope the value (as defined by @**OPTIMISE**) of every possible MMU position is evaluated and if it is above Head Grade then it is flagged as economic. All economic MMUs are sorted by value. The MMU with the maximum value is selected and all sub-MMUs within that MMU are assigned as Ranked. The next best MMU is then selected and those sub-MMUs that are not already assigned as Ranked are identified and evaluated. If the sum of the incremental sub-MMUs is above Head Grade then the incremental sub-MMUs are flagged as Ranked. This procedure is repeated until all economic MMUs in the sorted list have been processed. As with the calculation of the Minimum envelope a sub-MMU that was initially assigned as Maximum may be reassigned as Ranked.

As can be seen from the above definitions all three envelope types use the Head Grade to define an economic MMU but only the Minimum envelope uses the Cutoff Grade, to define the boundary between ore and waste. In general it is suggested that the Head Grade and Cutoff Grade are made equal. However if the Head Grade is calculated on the basis that it carries all costs but the Cutoff Grade represents the breakeven operating cost then Cutoff Grade can be made less than Head Grade. This will allow marginal ore to be included in the envelope although this could result in the grade of an envelope being below Head Grade but above Cutoff Grade.

The Ranked envelope gives a slightly smaller tonnage, but higher average grade, than the Minimum envelope and will maximize the value of the deposit for a given Head Grade. The Ranked envelope algorithm ensures that the grade of a Ranked envelope will always be above Head Grade. Unless the option of using a Cutoff Grade less than the Head Grade is appropriate it is suggested that the Ranked envelope is selected (@**ENVTYPE** =2) rather than the Minimum envelope (@**ENVTYPE** =1).

The Maximum envelope provides an outer envelope encompassing all acceptable alternate MMU positions that contain and could be considered to mine the ore blocks. Detailed mine design should be based around the Ranked or Minimum envelope and be within the Maximum envelope. The increment between the Ranked or Minimum envelope and the Maximum envelope includes marginal ore that can be used if the design needs to include additional material.

### Envelope Numbering

If @**ENVNUM** =1 then envelopes of contiguous sub-MMUs will be identified and each envelope will be assigned a unique envelope number. Sub-MMUs are considered to be contiguous if they share a common face but are not contiguous if they only share a common edge. Envelope numbers are assigned sequentially starting at 1 and incrementing by 1. The Results file then includes an evaluation of each envelope.

If @**ENVNUM** =0 then all sub-MMUs within all envelopes are assigned envelope number 1 (field **ENVNUM** =1) so the results file is just the total evaluation over all envelopes. Setting @**ENVNUM** =0 makes processing faster.

### Post-Processing Waste

An optional waste post-processing step, controlled by parameter @**OPTWASTE** , is used to determine whether a waste sub-MMU that is enclosed by or adjacent to a mineable envelope should be included as part of the envelope. A test is made to determine how many MMUs which include that sub-MMU meet the Head Grade criteria. This is expressed as a fraction of the total number of MMUs that include the sub-MMU. If this fraction is greater than or equal to the @**OPTWASTE** parameter then the sub-MMU is included as part of the mineable envelope. If this post-processing option is not required then the value of the parameter should be set to zero which is the default.

### Input Files:

* **in** (Block Model):
  Input model file for evaluation. This must have the fields **XMORIG** , **YMORIG** ,
  **ZMORIG** , **NX** , **NY** , **NZ** (implicit) and **IJK** , **XC** , **YC** and
  **ZC** (explicit). **XINC** , **YINC** and **ZINC** must exist as either explicit
  (sub-cells permitted) or implicit (no sub-cells). There must also be at least one
  explicit numeric data field, to be specified as * **VALUE** or * **GRADE**. The records
  may be in any order, but speed is increased if they are in IJK order. If it is a Rotated
  Model then it must also include the fields **X0, Y0, Z0, ANGLE1, ANGLE2, ANGLE3,
  ROTAXIS1, ROTAXIS2** , and **ROTAXIS3**.
  Required=Yes

* **shape** (Block Model):
  Input envelope shape template file to define one or more envelope shapes, or the
  orientation of the default envelope shape. Must contain fields **XC, YC, ZC, XINC, YINC,
  ZINC, XMORIG, YMORIG, ZMORIG, NX, NY, NZ, IJK** and optionally * **ZONE**. If the
  envelope orientation is not parallel to the input model then this model file must be a
  Rotated Model that include the fields **X0, Y0, Z0, ANGLE1, ANGLE2, ANGLE3, ROTAXIS1,
  ROTAXIS2** , and **ROTAXIS3**.
  Required=No

* **exclude** (Undefined):
  Optional input file to supply those values of one field in the input model that define
  an area for exclusion from the envelope. A maximum of 50 values is allowed. The field
  name in this file should be the same as a field in the input model file.
  Required=No

### Output Files:

* **out** (Block Model):
  Output model file with the additional field *MINED. May be the same as &IN if no new
  fields are created.
  Required=No

* **envmodel** (Block Model):
  Output model with the envelope grade distribution, where the envelope mimimum mining
  unit increment defines the cell size. The file is a standard model file with the fields
  ***VALUE, *GRADE, *ENVBEST, *DENSITY, *ENVELOPE** and * **SHAPZONE** if specified. The
  value in the * **ENVBEST** field depends on the value of @**OPTIMISE** :
  Required=No

* **results** (Undefined):
  Output results file to report statistics for each envelope, with fields ***ENVNUM,
  *ENVELOPE, *SHAPZONE** (if specified), ***VALUE, *GRADE, VOLUME, TONNES, MINX, MAXX,
  MINY, MAXY, MINZ, MAXZ, COGX, COGY, COGZ. *ENVELOPE** and ***SHAPZONE** fields are
  reported individually and in total.
  Required=No

### Fields:

* **value** (Numeric : IN):
  Numeric (explicit) field for the value of input model blocks.
  Default=Undefined
  Required=No

* **grade** (Numeric : IN):
  Numeric (explicit) field for the grade of input model blocks.
  Default=Undefined
  Required=Yes

* **shapzone** (Any : IN, SHAPE):
  Field in the input model and envelope shape template file when different envelope
  shapes are allowed in different mining areas.
  Default=Undefined
  Required=No

* **envbest** (Numeric : Undefined):
  Numeric (explicit) field for the best envelope grade or value in the &**ENVMODEL**
  file.
  Default=ENVBEST
  Required=No

* **envelope** (Character : Undefined):
  Alphanumeric (explicit) field for the cell type in the &**ENVMODEL** file. The default
  name **ENVELOPE** is used if none is supplied.
  Default=ENVELOPE
  Required=No

* **envnum** (Numeric : Undefined):
  Numeric (explicit) field for the envelope number in the &**ENVMODEL** and &**RESULTS**
  files. The default name **ENVNUM** is used if none is supplied.
  Default=ENVNUM
  Required=No

* **hdgrade** (Numeric : IN):
  Optional field for the head grade of envelopes in which the block participates. The
  envelope head grade must be greater than the head grade value supplied from any
  participating block. Absent values are ignored.
  Default=Undefined
  Required=No

* **mined** (Numeric : Undefined):
  Proportion of block within the envelope
  Default=MINED
  Required=No

* **density** (Numeric : IN):
  Optional density field in the input model for average grade and tonnage calculations.
  Default=DENSITY
  Required=No

### Parameters:

* **cutoff**:
  Cutoff grade to be applied to input model blocks.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **hdgrade**:
  Required head grade for economic envelopes.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **defgrade**:
  Default grade for envelope volume not modelled with blocks, or blocks with an absent
  grade or value.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **defvalue**:
  Default value for envelope volume not modelled with blocks, or blocks with an absent
  grade or value. This parameter is expressed as value per unit volume.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **density**:
  Density value where a density field is not supplied, the value is absent, or the
  envelope volume is not modelled with blocks. If set to zero then absent blocks are
  assumed to be air.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **maxwaste**:
  Maximum waste material in an envelope, expressed as a volume fraction, for an envelope
  evaluation to be accepted as an alternative in the optimal selection. The default (1.0)
  allows any proportion of waste provided the head grade target is met. The value cannot
  be lower than the @**PDWASTE** value.
  Range=0,1
  Values=Undefined
  Default=1
  Required=Yes

* **maxore**:
  Maximum ore material in a pre-defined waste shape, expressed as a volume fraction.
  Range=0,1
  Values=Undefined
  Default=0
  Required=Yes

* **pdwvalue**:
  Maximum value for a pre-defined waste shape.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **pdwgrade**:
  Maximum grade for a pre-defined waste shape.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **pdwaste**:
  Maximum pre-defined waste material in an envelope, expressed as a volume fraction, for
  an envelope evaluation to be accepted as an alternative in the optimal selection. The
  default value of 0 allows no pre-defined waste, and a value between 0 and 1 allows
  pre-defined waste to be taken as a last resort to extract otherwise economic ore.
  Range=0,1
  Values=Undefined
  Default=0
  Required=No

* **optwaste**:
  Post-process remnant waste "internal" to the envelope that is not flagged as
  pre-defined waste or already included in a mining envelope to evaluate if the waste can
  be in one, some, or all alternative envelopes. The proportion is specified as a
  fraction, and will only be processed for a non-zero value. Only those blocks that are
  outside the minimum envelope but included in the maximum envelope are considered. A
  value of zero would generate the maximum envelope.
  Range=0,1
  Values=Undefined
  Default=0
  Required=No

* **optimise**:
  Method for selecting optimal envelope position where alternate positions are available
  to be considered:
  Range=
  Values=
  Default=
  Required=No

* **dilute**:
  Include waste with ore in envelope grade calculations, and * **ENVBEST** output:
  Range=
  Values=
  Default=
  Required=No

* **envtype**:
  Report the minimum or maximum envelope in the results file for sequencing. Both minimum
  and maximum envelopes are generated in the optimiser.
  Range=
  Values=
  Default=
  Required=No

* **envnum**:
  Apply a unique numbering scheme to the envelopes. Having unique numbers can slow down
  processing by [very] approximately 20%.
  Range=
  Values=
  Default=
  Required=No

* **inmodel**:
  Constrain envelopes to the volume of the input model occupied by blocks:
  Range=
  Values=
  Default=
  Required=No

* **shapemth**:
  Method for selection from alternative envelope shapes when zones are specified:
  Range=
  Values=
  Default=
  Required=No

* **pdwonly**:
  Flag to specify whether the current run should create both predefined waste and mining
  envelopes, or only the pre-defined waste envelope:
  Range=
  Values=
  Default=
  Required=No

* **mmuincx**:
  Number of envelope increments within the minimum envelope dimension in X coordinate.
  Range=1,+
  Values=Undefined
  Default=1
  Required=Yes

* **mmuincy**:
  Number of envelope increments within the minimum envelope dimension in Y coordinate.
  Range=1,+
  Values=Undefined
  Default=1
  Required=Yes

* **mmuincz**:
  Number of envelope increments within the minimum envelope dimension in Z coordinate.
  Range=1,+
  Values=Undefined
  Default=1
  Required=Yes

* **mmusizex**:
  Minimum envelope dimension in the horizontal X coordinate.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **mmusizey**:
  Minimum envelope dimension in the horizontal Y coordinate.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **mmusizez**:
  Minimum envelope dimension in the vertical Z coordinate.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **pdwsizex**:
  Minimum pre-defined waste shape dimension in the horizontal X coordinate.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **pdwsizey**:
  Minimum pre-defined waste shape dimension in the horizontal Y coordinate.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **pdwsizez**:
  Minimum pre-defined waste shape dimension in the vertical Z coordinate.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **mmuslopn**:
  Envelope slope factor 1:n in the northerly direction, positive outwards
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mmuslops**:
  Envelope slope factor 1:n in the southerly direction, positive outwards
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mmuslope**:
  Envelope slope factor 1:n in the easterly direction, positive outwards
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mmuslopw**:
  Envelope slope factor 1:n in the westerly direction, positive outwards
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mmuslopi**:
  Specifies if the slope factors are to be interpolated between orthogonal directions 0 -
  rectangular horizontal shape 1 - elliptical horizontal shape
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mmuminx**:
  Minimum X coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmuminy**:
  Minimum Y coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmuminz**:
  Minimum Z coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmumaxx**:
  Maximum X coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmumaxy**:
  Maximum Y coordinate for envelope volume. This is not required if an envelope shape
  template file &SHAPE is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmumaxz**:
  Maximum Z coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmumaxz**:
  Maximum Z coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmumaxz**:
  Maximum Z coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **mmumaxz**:
  Maximum Z coordinate for envelope volume. This is not required if an envelope shape
  template file &**SHAPE** is defined.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xsubcell**:
  Number of subcells per parent cell in X direction
  Range=1,100
  Values=Undefined
  Default=1
  Required=No

* **ysubcell**:
  Number of subcells per parent cell in Y direction
  Range=1,100
  Values=Undefined
  Default=1
  Required=No

* **zsubcell**:
  Number of subcells per parent cell in Z direction
  Range=1,100
  Values=Undefined
  Default=1
  Required=No

* **envout**:
  Flag to control which **ENVELOPE** types are included in the output envelope file
  &**ENVMODEL** : Options: 0: Include all values of the field **ENVELOPE**; 1: Include all
  values of the field **ENVELOPE** except _UND_; 2: Include all values of the field
  **ENVELOPE** except _UND_ , _MOD. EXC_; 3: Include all values of the field **ENVELOPE**
  except UND, _MOD, EXC, PDW_; 4: Include all values of the field **ENVELOPE** except

## _UND, MOD, EXC, PDW, MAX-UND, MAX-MOD, MAX-PDW_

  Range=0,4
  Values=0,1,2,3,4
  Default=1
  Required=No

* **xoverlap**:
  Overlap in X between successive slices, defined as the number of MMUs in X. ie in
  length units the overlap is @**XOVERLAP** * @**MMUSIZEX** : Options: 1: \- overlap of 1
  MMU widths in X; 2: \- overlap of 2 MMU widths in X
  Range=1,2
  Values=1,2
  Default=2
  Required=No

* **calcenv**:
  Flag to select either a test run to report slicing and memory statistics or a full run
  to calculate the mineable envelopes: Options: 0: \- test run; do not calculate mineable
  envelopes; 1: \- full run; calculate mineable envelopes
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **progress**:
  Progress counter increment for progress messages displayed in the bottom right corner
  of the Studio window. Increasing the increment can reduce processing time.
  Range=Undefined
  Values=UNdefined
  Default=5000
  Required=No

* **info**:
  Flag to control the level of information displayed to the Output Window during

* **processing** (Options: 1: Minimum level of output; 2: Intermediate level of output; 3:):
  Maximum level of output
  Range=1,3
  Values=1,2,3
  Default=2
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('modenv')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute modenv
print("Running modenv...")
dm_cmd.modenv(
    in_i='t_assays',  # required
    value_f='optional',  # required
    grade_f='optional',  # required
    envelope_f='optional',  # required
    hdgrade_f='optional',  # required
    mined_f='optional',  # required
    density_f='optional',  # required
    cutoff_p='required_val',  # required
    hdgrade_p='required_val',  # required
    density_p=1,  # required
    maxwaste_p='required_val',  # required
    maxore_p='required_val',  # required
    pdwvalue_p='required_val',  # required
    pdwgrade_p='required_val',  # required
    pdwonly_p='optional',  # required
    mmuincx_p='required_val',  # required
    mmuincy_p='required_val',  # required
    mmuincz_p='required_val',  # required
    mmusizex_p='required_val',  # required
    mmusizey_p='required_val',  # required
    mmusizez_p='required_val',  # required
    pdwsizex_p='required_val',  # required
    pdwsizey_p='required_val',  # required
    pdwsizez_p='required_val',  # required
    # shape_i='optional',  # optional
    # exclude_i='optional',  # optional
    # out_o='t_modenv_out',  # optional
    # envmodel_o='t_modenv_out',  # optional
    # results_o=['t_modenv_out'],  # optional
    # shapzone_f='optional',  # optional
    # envbest_f='optional',  # optional
    # envnum_f='optional',  # optional
    # defgrade_p=0,  # optional
    # defvalue_p=0,  # optional
    # pdwaste_p=0,  # optional
    # optwaste_p=0,  # optional
    # optimise_p='optional',  # optional
    # dilute_p='optional',  # optional
    # envtype_p='optional',  # optional
    # envnum_p='optional',  # optional
    # inmodel_p='optional',  # optional
    # shapemth_p='optional',  # optional
    # mmuslopn_p=0,  # optional
    # mmuslops_p=0,  # optional
    # mmuslope_p=0,  # optional
    # mmuslopw_p=0,  # optional
    # mmuslopi_p=0,  # optional
    # mmuminx_p='optional',  # optional
    # mmuminy_p='optional',  # optional
    # mmuminz_p='optional',  # optional
    # mmumaxx_p='optional',  # optional
    # mmumaxy_p='optional',  # optional
    # mmumaxz_p='optional',  # optional
    # mmumaxz2_p='optional',  # optional
    # mmumaxz3_p='optional',  # optional
    # mmumaxz4_p='optional',  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # zsubcell_p=1,  # optional
    # envout_p=1,  # optional
    # xoverlap_p=2,  # optional
    # calcenv_p=1,  # optional
    # progress_p=5000,  # optional
    # info_p=2,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("modenv execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_modenv_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")